In [27]:
import time
import torch
import pandas as pd

from pathlib import Path
from PIL import Image

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# 경로 설정
DATA_DIR = Path("../data")

ORIGINAL_DIR = DATA_DIR / "train_images"
CACHE_DIR = DATA_DIR / "train_images_224"

print("원본 이미지 경로:", ORIGINAL_DIR.resolve())
print("캐시 이미지 경로:", CACHE_DIR.resolve())

print("원본 폴더 존재:", ORIGINAL_DIR.exists())
print("캐시 폴더 존재:", CACHE_DIR.exists())

원본 이미지 경로: C:\Users\SSAFY\Desktop\ParkJaeWon\AI-practice\data\train_images
캐시 이미지 경로: C:\Users\SSAFY\Desktop\ParkJaeWon\AI-practice\data\train_images_224
원본 폴더 존재: True
캐시 폴더 존재: True


In [28]:
train_data = pd.read_csv(DATA_DIR / "train_split.csv")
valid_data = pd.read_csv(DATA_DIR / "valid_split.csv")

print("Train:", train_data.shape)
print("Valid:", valid_data.shape)

display(train_data.head())

Train: (14906, 2)
Valid: (3726, 2)


,image,labels
0,800113bb65efe69e.jpg,healthy
1,8002cb321f8bfcdf.jpg,scab frog_eye_leaf_spot complex
2,80070f7fb5e2ccaa.jpg,scab
3,80077517781fb94f.jpg,scab
4,800edef467d27c15.jpg,healthy


In [29]:
classes = [
    "scab",
    "healthy",
    "frog_eye_leaf_spot",
    "complex",
    "rust",
    "powdery_mildew"
]

def multi_hot_encode(labels):
    label_list = labels.split()

    encoded = []

    for cls in classes:
        if cls in label_list:
            encoded.append(1)
        else:
            encoded.append(0)

    return encoded


# 테스트
print(multi_hot_encode("scab rust"))

[1, 0, 0, 0, 1, 0]


In [30]:
class PlantDataset(Dataset):

    def __init__(self, df, image_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_path = self.image_dir / row["image"]

        with Image.open(image_path) as image:
            image = image.convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(
            multi_hot_encode(row["labels"]),
            dtype=torch.float32
        )

        return image, label

In [16]:

from torch.utils.data import Dataset, DataLoader

class PlanetDataset(Dataset):
    def __init__(self, df, image_dir, transform = None):
        self.df = df.reset_index(drop = True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx): # Dataset에서 이미지 한 장을 요청하면 해당 이미지와 정답을 반환하는 역할
        row = self.df.iloc[idx]

        image_path = self.image_dir / row["image"]
        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(
            multi_hot_encode(row["labels"]),
            dtype = torch.float32
        )

        return image, label

In [31]:
# 원본 이미지용
original_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# 캐시 이미지용
cache_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [32]:
# 원본 Train Dataset
train_dataset = PlantDataset(
    train_data,
    ORIGINAL_DIR,
    transform=original_transform
)

# 캐시 Train Dataset
cache_dataset = PlantDataset(
    train_data,
    CACHE_DIR,
    transform=cache_transform
)

# 캐시 Validation Dataset
valid_dataset = PlantDataset(
    valid_data,
    CACHE_DIR,
    transform=cache_transform
)

print("원본 Train:", len(train_dataset))
print("캐시 Train:", len(cache_dataset))
print("캐시 Valid:", len(valid_dataset))

원본 Train: 14906
캐시 Train: 14906
캐시 Valid: 3726


In [33]:
BATCH_SIZE = 32

# 원본 이미지
original_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

# 캐시 이미지
cache_loader = DataLoader(
    cache_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

# Validation
valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

In [34]:
images, labels = next(iter(cache_loader))

print("Images:", images.shape)
print("Labels:", labels.shape)
print("Labels dtype:", labels.dtype)

print("첫 번째 이미지 라벨:", labels[0])

Images: torch.Size([32, 3, 224, 224])
Labels: torch.Size([32, 6])
Labels dtype: torch.float32
첫 번째 이미지 라벨: tensor([0., 1., 0., 0., 0., 0.])


In [35]:
def measure_loader(loader, num_batches=20):

    start = time.perf_counter()
    count = 0

    for images, labels in loader:
        count += 1

        if count >= num_batches:
            break

    elapsed = time.perf_counter() - start

    return elapsed / count

In [36]:
original_time = measure_loader(
    original_loader,
    num_batches=20
)

cache_time = measure_loader(
    cache_loader,
    num_batches=20
)

print(f"원본 이미지: {original_time:.3f}초 / batch")
print(f"캐시 이미지: {cache_time:.3f}초 / batch")
print(f"속도 개선: {original_time / cache_time:.2f}배")

원본 이미지: 2.880초 / batch
캐시 이미지: 0.053초 / batch
속도 개선: 54.41배


In [37]:
original_paths = list(ORIGINAL_DIR.glob("*.jpg"))
cache_paths = list(CACHE_DIR.glob("*.jpg"))

print("원본 이미지:", len(original_paths))
print("캐시 이미지:", len(cache_paths))

original_names = {p.name for p in original_paths}
cache_names = {p.name for p in cache_paths}

print("누락된 이미지:", len(original_names - cache_names))

if cache_paths:
    with Image.open(cache_paths[0]) as image:
        print("캐시 이미지 크기:", image.size)

원본 이미지: 18632
캐시 이미지: 18632
누락된 이미지: 0
캐시 이미지 크기: (224, 224)
